In [1]:
import numpy as np
import skimage as ski
import napari
from magicgui.widgets import PushButton, Container, Label
from pathlib import Path

In [2]:
projections_dir = Path('Data/projections')
sources = sorted(p for p in projections_dir.glob('*.tif') if not p.stem.endswith('_ring'))

def labels_path_for(src):
    return src.with_name(src.stem + '_ring.tiff')

current = [next((i for i, s in enumerate(sources) if not labels_path_for(s).exists()), 0)]

In [3]:
viewer = napari.Viewer()
status = Label(value='')

def load_current():
    src = sources[current[0]]
    img = ski.io.imread(src)
    viewer.layers.clear()
    viewer.add_image(img, name=src.name)
    lp = labels_path_for(src)
    labels_data = ski.io.imread(lp) if lp.exists() else np.zeros(img.shape, dtype=np.uint8)
    viewer.add_labels(labels_data, name='ring')
    status.value = f'[{current[0]+1}/{len(sources)}] {src.name}'

def save_and_advance():
    src = sources[current[0]]
    ski.io.imsave(labels_path_for(src), viewer.layers['ring'].data.astype(np.uint8), check_contrast=False)
    if current[0] + 1 >= len(sources):
        status.value = 'done'
        return
    current[0] += 1
    load_current()

next_btn = PushButton(text='Save & Next')
next_btn.changed.connect(lambda *_: save_and_advance())
panel = Container(widgets=[next_btn, status])
viewer.window.add_dock_widget(panel, area='right')

load_current()